# Notebook 3 — Concurrency & Object-Oriented Programming

**Topics covered:** Multi-threading · Asynchronous Programming · OOP Fundamentals · Inheritance · Encapsulation & Polymorphism

| Topic | Subtopics |
|---|---|
| 1 · Multi-threading | Thread creation, Lock synchronization, Event coordination |
| 2 · Async Programming | Coroutines, `asyncio.gather()`, threading vs asyncio |
| 3 · OOP Fundamentals | Class & `__init__`, methods & `@property`, dunder methods, class/static methods |
| 4 · Inheritance | Single inheritance, method overriding, multiple inheritance & MRO |
| 5 · Encapsulation & Polymorphism | `_`/`__` naming, duck typing & polymorphism |

---

### Sample Data
All examples use two consistent person records — run the setup cell below before any other cell.

In [ ]:
# Shared sample data used throughout this notebook
people = [
    {
        "first_name": "Hanu",
        "last_name":  "Madala",
        "dob":        "20 Jan 2002",
        "phone":      "+65 1234 5678",
        "email":      "hanu.m@tdfs.com",
    },
    {
        "first_name": "Shankar",
        "last_name":  "Cherukuri",
        "dob":        "30 Mar 1997",
        "phone":      "98765432",
        "email":      "shankar.c@tdfs.com",
    },
]

---
## Topic 1 — Multi-threading

Python's `threading` module lets you run multiple tasks concurrently within the same process. Threads share memory, making synchronization essential when they access shared state.

#### 1a · Thread creation — `threading.Thread`, `target`, `start()`, `join()`

Pass a callable to `threading.Thread(target=...)`, call `.start()` to launch it and `.join()` to block the main thread until it finishes.

In [ ]:
import threading
import time

def print_details(person, delay, count=3):
    for i in range(count):
        print(f"[Thread] {person['first_name']} {person['last_name']} — round {i + 1}")
        time.sleep(delay)

t1 = threading.Thread(target=print_details, args=(people[0], 0.3))
t2 = threading.Thread(target=print_details, args=(people[1], 0.5))

t1.start()
t2.start()

t1.join()
t2.join()

print("Both threads finished.")

#### 1b · Thread synchronization — `Lock`, `acquire()`, `release()`, `with lock`

A `Lock` ensures only one thread enters a critical section at a time, preventing race conditions on shared state.

In [ ]:
import threading

counter = 0
lock    = threading.Lock()

def increment(person, times=4):
    global counter
    for _ in range(times):
        with lock:                  # acquire on entry, release on exit
            counter += 1
            print(f"  {person['first_name']:8s} → counter = {counter}")

t1 = threading.Thread(target=increment, args=(people[0],))
t2 = threading.Thread(target=increment, args=(people[1],))
t1.start(); t2.start()
t1.join();  t2.join()

print(f"\nFinal counter: {counter}  (expected 8)")

#### 1c · Thread coordination — `Event`, `set()`, `wait()`

An `Event` acts as a flag: one thread calls `.set()` to signal readiness and another calls `.wait()` to block until the flag is raised.

In [ ]:
import threading
import time

data_ready = threading.Event()

def prepare_data(person):
    print(f"[Producer] Preparing data for {person['first_name']}...")
    time.sleep(0.4)
    print(f"[Producer] Data ready for {person['first_name']}.")
    data_ready.set()               # signal the consumer

def process_data(person):
    print("[Consumer] Waiting for data...")
    data_ready.wait()              # block until set()
    print(f"[Consumer] Processing {person['first_name']} {person['last_name']}'s record.")

t_producer = threading.Thread(target=prepare_data, args=(people[0],))
t_consumer = threading.Thread(target=process_data, args=(people[0],))

t_consumer.start()     # consumer starts first, waits for event
t_producer.start()
t_producer.join()
t_consumer.join()

---
## Topic 2 — Asynchronous Programming

`asyncio` provides single-threaded cooperative concurrency: coroutines yield control at `await` points, allowing other coroutines to run while waiting for I/O.

#### 2a · Coroutine definition — `async def`, `await`, `asyncio.sleep()`

`async def` defines a coroutine function; `await` suspends it and yields control back to the event loop until the awaited task completes.

In [ ]:
import asyncio

async def fetch_profile(person):
    """Simulate an async API call with a network delay."""
    print(f"  Fetching profile for {person['first_name']}...")
    await asyncio.sleep(0.5)       # non-blocking wait (simulates network I/O)
    return f"{person['first_name']} {person['last_name']} ({person['email']})"

# In a Jupyter notebook the event loop is already running — use await directly
result = await fetch_profile(people[0])
print(result)

#### 2a (real I/O) · Async HTTP requests — `aiohttp.ClientSession`, `session.get()`

`aiohttp` provides an async-native HTTP client; each `session.get()` call suspends the coroutine at the `await` point while the network response arrives, freeing the event loop to run other coroutines.

In [ ]:
import asyncio
import aiohttp

async def lookup_name(session, person):
    """
    Call agify.io — a free public API that predicts age from a first name.
    Returns the raw JSON response dict for that person.
    """
    url = f"https://api.agify.io?name={person['first_name']}"
    async with session.get(url) as response:
        data = await response.json()
        return {
            "person":      person["first_name"],
            "predicted_age": data.get("age"),
            "sample_count":  data.get("count"),
        }

async def fetch_all():
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[lookup_name(session, p) for p in people])
    return results

results = await fetch_all()
for r in results:
    print(f"{r['person']:10s} → predicted age {r['predicted_age']}  "
          f"(based on {r['sample_count']:,} records)")

#### 2b · Concurrent coroutines — `asyncio.gather()`

`asyncio.gather()` schedules multiple coroutines to run concurrently and collects all their return values in order.

In [ ]:
import asyncio
import time

start = time.perf_counter()

# Both coroutines run concurrently — total time ≈ max(delay) not sum(delays)
results = await asyncio.gather(
    fetch_profile(people[0]),
    fetch_profile(people[1]),
)

elapsed = time.perf_counter() - start
print(f"Results (in {elapsed:.2f}s):")
for r in results:
    print(f"  {r}")

#### 2c · Threading vs asyncio — I/O-bound vs CPU-bound comparison

Use `asyncio` for async-native I/O (HTTP, WebSockets); use `threading` for blocking I/O (legacy APIs, file reads); use `multiprocessing` for CPU-bound compute.

In [ ]:
# Side-by-side timing comparison
import asyncio
import threading
import time

DELAY = 0.3

# --- asyncio approach ---
async def async_task(name, delay):
    await asyncio.sleep(delay)
    return f"[async]  {name} done"

t0 = time.perf_counter()
async_results = await asyncio.gather(
    async_task(people[0]['first_name'], DELAY),
    async_task(people[1]['first_name'], DELAY),
)
async_time = time.perf_counter() - t0

# --- threading approach ---
thread_results = {}

def thread_task(name, delay, key):
    time.sleep(delay)
    thread_results[key] = f"[thread] {name} done"

t0 = time.perf_counter()
threads = [
    threading.Thread(target=thread_task, args=(people[0]['first_name'], DELAY, 0)),
    threading.Thread(target=thread_task, args=(people[1]['first_name'], DELAY, 1)),
]
for t in threads: t.start()
for t in threads: t.join()
thread_time = time.perf_counter() - t0

print("asyncio:",  *async_results,          f"({async_time:.2f}s)", sep="\n  ")
print("threading:", *thread_results.values(), f"({thread_time:.2f}s)", sep="\n  ")

---
## Topic 3 — OOP Fundamentals

A class is a blueprint for objects. Each instance carries its own state (attributes) and behaviour (methods). Python exposes special hooks through dunder methods (`__init__`, `__repr__`, etc.).

#### 3a · Class definition — `class`, `__init__`, instance attributes

`__init__` is the initialiser called when an instance is created; `self` refers to the new object and is used to attach instance attributes.

In [ ]:
class Person:
    def __init__(self, first_name, last_name, dob, phone, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.phone      = phone
        self.email      = email

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")

print(hanu.first_name, hanu.email)
print(shankar.last_name, shankar.dob)

#### 3b · Instance methods and `@property`

Instance methods receive `self` implicitly; `@property` wraps a method so it can be accessed as an attribute without parentheses.

In [ ]:
from datetime import datetime

class Person:
    def __init__(self, first_name, last_name, dob, phone, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.phone      = phone
        self.email      = email

    @property
    def full_name(self):
        """Computed attribute — accessed without ()."""
        return f"{self.first_name} {self.last_name}"

    def age(self):
        """Returns the person's current age in full years."""
        dob_date = datetime.strptime(self.dob, "%d %b %Y")
        today    = datetime.today()
        return today.year - dob_date.year - (
            (today.month, today.day) < (dob_date.month, dob_date.day)
        )

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")

print(hanu.full_name, "—", hanu.age(), "years old")       # property, no ()
print(shankar.full_name, "—", shankar.age(), "years old")

#### 3c · Dunder methods — `__repr__`, `__str__`

`__repr__` returns an unambiguous developer-facing string (used by `repr()` and in the REPL); `__str__` returns a human-readable string (used by `print()` and `str()`).

In [ ]:
class Person:
    def __init__(self, first_name, last_name, dob, phone, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.phone      = phone
        self.email      = email

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

    def age(self):
        from datetime import datetime
        dob_date = datetime.strptime(self.dob, "%d %b %Y")
        today    = datetime.today()
        return today.year - dob_date.year - (
            (today.month, today.day) < (dob_date.month, dob_date.day))

    def __repr__(self):
        return f"Person(name={self.full_name!r}, dob={self.dob!r})"

    def __str__(self):
        return f"{self.full_name} (age {self.age()}, {self.email})"

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")

print(repr(hanu))    # → Person(name='Hanu Madala', dob='20 Jan 2002')
print(str(shankar))  # → Shankar Cherukuri (age 28, shankar.c@tdfs.com)

#### 3d · Class methods and static methods — `@classmethod`, `@staticmethod`

`@classmethod` receives the class itself (`cls`) as the first argument and can access or modify class-level state; `@staticmethod` is a plain function scoped inside the class for organisational reasons.

In [ ]:
class Person:
    _registry = []          # class-level shared state

    def __init__(self, first_name, last_name, dob, phone, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.phone      = phone
        self.email      = email
        Person._registry.append(self)

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

    @classmethod
    def registry(cls):
        """Return full names of all registered Person instances."""
        return [p.full_name for p in cls._registry]

    @staticmethod
    def is_international_phone(phone):
        """Returns True if the phone string starts with '+'."""
        return phone.startswith("+")

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")

print(Person.registry())
print(Person.is_international_phone(hanu.phone))     # True
print(Person.is_international_phone(shankar.phone))  # False

---
## Topic 4 — Inheritance

Inheritance lets a subclass reuse and extend a parent class. Python supports single and multiple inheritance, with method resolution order (MRO) determining which method runs when names conflict.

#### 4a · Single inheritance — `super()`, extending a base class

`super()` delegates to the next class in the MRO, allowing a subclass to call its parent's `__init__` or methods without hard-coding the parent name.

In [ ]:
class Employee(Person):
    def __init__(self, first_name, last_name, dob, phone, email,
                 employee_id, department):
        super().__init__(first_name, last_name, dob, phone, email)  # delegate to Person
        self.employee_id = employee_id
        self.department  = department

    def badge(self):
        return f"{self.employee_id} — {self.full_name} ({self.department})"

emp = Employee(
    "Hanu", "Madala", "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com",
    "EMP-001", "Engineering"
)

print(emp.badge())
print(emp.email)              # inherited from Person
print(isinstance(emp, Person))  # True — Employee IS-A Person

#### 4b · Method overriding — redefining parent behaviour in a subclass

When a subclass defines a method with the same name as the parent, the subclass version runs; `super()` can still be used to invoke the parent version explicitly.

In [ ]:
class Employee(Person):
    def __init__(self, first_name, last_name, dob, phone, email,
                 employee_id, department):
        super().__init__(first_name, last_name, dob, phone, email)
        self.employee_id = employee_id
        self.department  = department

    def badge(self):
        return f"{self.employee_id} — {self.full_name} ({self.department})"

    def __str__(self):
        # Overrides Person.__str__ — adds employee_id context
        return f"[{self.employee_id}] {self.full_name} — {self.department}"

    def __repr__(self):
        return f"Employee(id={self.employee_id!r}, name={self.full_name!r})"

emp = Employee(
    "Shankar", "Cherukuri", "30 Mar 1997", "98765432", "shankar.c@tdfs.com",
    "EMP-002", "Data Science"
)

print(str(emp))   # uses Employee.__str__
print(repr(emp))  # uses Employee.__repr__

#### 4c · Multiple inheritance and MRO — mixins, `__mro__`

Python resolves method lookup using the C3 linearisation algorithm (MRO). Mixin classes add reusable behaviour without representing a full base type.

In [ ]:
class Auditable:
    def audit_log(self):
        return f"[AUDIT] {getattr(self, 'full_name', '?')} record accessed"

class ContactMixin:
    def contact_card(self):
        return f"{getattr(self, 'email', '?')} | {getattr(self, 'phone', '?')}"

class Staff(Employee, Auditable, ContactMixin):
    pass

staff = Staff(
    "Hanu", "Madala", "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com",
    "EMP-001", "Engineering"
)

print(staff.badge())          # from Employee
print(staff.audit_log())      # from Auditable
print(staff.contact_card())   # from ContactMixin

print("\nMRO:")
for cls in Staff.__mro__:
    print(f"  {cls.__name__}")

---
## Topic 5 — Encapsulation & Polymorphism

Encapsulation bundles data and hides implementation details; polymorphism allows different types to be used interchangeably when they share the same interface.

#### 5a · Encapsulation — `_` single and `__` double underscore name mangling

`_name` is a convention signalling internal use; `__name` is name-mangled by the interpreter to `_ClassName__name`, providing stronger (though not absolute) access restriction.

In [ ]:
class SecurePerson(Person):
    def __init__(self, first_name, last_name, dob, phone, email, password):
        super().__init__(first_name, last_name, dob, phone, email)
        self._phone     = phone        # _single: internal — callers should use masked_phone
        self.__password = password     # __double: name-mangled to _SecurePerson__password

    def verify(self, attempt):
        return attempt == self.__password

    @property
    def masked_phone(self):
        return "****" + self._phone[-4:]

hanu_s = SecurePerson(
    "Hanu", "Madala", "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com", "s3cr3t"
)

print(hanu_s.masked_phone)              # ****5678
print(hanu_s.verify("s3cr3t"))          # True
print(hanu_s.verify("wrong"))           # False

# Direct __password access raises AttributeError:
try:
    print(hanu_s.__password)
except AttributeError as e:
    print(f"AttributeError: {e}")

# Accessible via mangled name (but strongly discouraged):
print(hanu_s._SecurePerson__password)

#### 5b · Subclass polymorphism — overriding a base-class method, uniform interface

When subclasses override a method defined on the base class, a single loop or function can call that method on a mixed list of instances and Python dispatches to each class's own version at runtime.

In [19]:
# Base class defines the contract — every entity must implement summary()
class Entity:
    def summary(self):
        raise NotImplementedError("Subclasses must implement summary()")


class Person(Entity):
    def __init__(self, first_name, last_name, dob, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.email      = email

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

    def summary(self):                              # overrides Entity.summary
        return f"Person  : {self.full_name} | {self.email}"


class Employee(Person):
    def __init__(self, first_name, last_name, dob, email, employee_id, department):
        super().__init__(first_name, last_name, dob, email)
        self.employee_id = employee_id
        self.department  = department

    def summary(self):                              # overrides Person.summary
        return f"Employee: [{self.employee_id}] {self.full_name} — {self.department}"


class Bot(Entity):
    def __init__(self, name):
        self.name = name

    def summary(self):                              # overrides Entity.summary
        return f"Bot     : {self.name} (automated)"


# Mixed list — three different types, one shared interface
entities = [
    Person(  "Hanu",    "Madala",    "20 Jan 2002", "hanu.m@tdfs.com"),
    Employee("Shankar", "Cherukuri", "30 Mar 1997", "shankar.c@tdfs.com", "EMP-002", "Data Science"),
    Bot("DataBot-3000"),
]

# Same call — Python dispatches to each class's own summary() at runtime
for entity in entities:
    print(entity.summary())

Person  : Hanu Madala | hanu.m@tdfs.com
Employee: [EMP-002] Shankar Cherukuri — Data Science
Bot     : DataBot-3000 (automated)


#### 5c · Polymorphism — duck typing and method overriding

Duck typing lets a function work with any object that has the expected attributes or methods, regardless of its actual type — "if it walks like a duck and quacks like a duck, it is a duck".

In [ ]:
class Bot:
    """A non-Person entity that shares the same interface."""
    def __init__(self, name):
        self.full_name = name

    def age(self):
        return 0   # bots don't age

    def __str__(self):
        return f"[Bot] {self.full_name} (age 0)"


def describe(entity):
    """Works on any object that has .full_name and .age() — duck typing."""
    return f"{entity.full_name} is {entity.age()} years old"


hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")
bot     = Bot("DataBot-3000")

# describe() accepts Person and Bot alike — polymorphism via duck typing
for entity in [hanu, shankar, bot]:
    print(describe(entity))

---
## Exercises

Complete each exercise in the cell below it. Reference solutions are hidden in the `<details>` blocks.

### Exercise 1 — Threads with interleaving output

Create two threads:
- Thread 1 prints Hanu's full name and DOB every **0.5 s** (3 times)
- Thread 2 prints Shankar's full name and DOB every **0.7 s** (3 times)

Start both threads and `.join()` them. Observe the interleaved output.

<details>
<summary>Solution</summary>

```python
import threading, time

def print_person(person, delay, count=3):
    for _ in range(count):
        print(f"{person['first_name']} {person['last_name']} — {person['dob']}")
        time.sleep(delay)

t1 = threading.Thread(target=print_person, args=(people[0], 0.5))
t2 = threading.Thread(target=print_person, args=(people[1], 0.7))
t1.start(); t2.start()
t1.join();  t2.join()
```

</details>

In [ ]:
# Exercise 1 — your code here

### Exercise 2 — Async profile fetcher

Write an `async` function `fetch_profile(person)` that:
1. Prints `"Fetching <first_name>..."`
2. Awaits `asyncio.sleep(0.8)` to simulate a network delay
3. Returns `"<full_name> | <email>"`

Use `asyncio.gather()` to fetch both Hanu and Shankar **concurrently** and print all results.

<details>
<summary>Solution</summary>

```python
import asyncio

async def fetch_profile(person):
    print(f"Fetching {person['first_name']}...")
    await asyncio.sleep(0.8)
    return f"{person['first_name']} {person['last_name']} | {person['email']}"

results = await asyncio.gather(*[fetch_profile(p) for p in people])
for r in results:
    print(r)
```

</details>

In [ ]:
# Exercise 2 — your code here

### Exercise 3 — Person class with full_name and age

Implement a `Person` class with:
- `__init__` taking `first_name`, `last_name`, `dob`, `phone`, `email`
- A `full_name` property returning `"<first> <last>"`
- An `age()` method returning the person's age in full years
- `__str__` returning `"<full_name> (age <age>)"`

Instantiate both Hanu and Shankar and print them.

<details>
<summary>Solution</summary>

```python
from datetime import datetime

class Person:
    def __init__(self, first_name, last_name, dob, phone, email):
        self.first_name = first_name
        self.last_name  = last_name
        self.dob        = dob
        self.phone      = phone
        self.email      = email

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"

    def age(self):
        dob_date = datetime.strptime(self.dob, "%d %b %Y")
        today    = datetime.today()
        return today.year - dob_date.year - (
            (today.month, today.day) < (dob_date.month, dob_date.day)
        )

    def __str__(self):
        return f"{self.full_name} (age {self.age()})"

hanu    = Person("Hanu",    "Madala",    "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com")
shankar = Person("Shankar", "Cherukuri", "30 Mar 1997", "98765432",      "shankar.c@tdfs.com")
print(hanu)
print(shankar)
```

</details>

In [ ]:
# Exercise 3 — your code here

### Exercise 4 — Employee subclass with badge

Create an `Employee` subclass of `Person` that:
- Adds `employee_id` and `department` attributes
- Implements `badge()` returning `"<employee_id> — <full_name>"`
- Overrides `__str__` to return `"[<employee_id>] <full_name> — <department>"`

Instantiate one `Employee` using Hanu's data (`EMP-001`, `"Engineering"`) and print their badge and string representation.

<details>
<summary>Solution</summary>

```python
class Employee(Person):
    def __init__(self, first_name, last_name, dob, phone, email,
                 employee_id, department):
        super().__init__(first_name, last_name, dob, phone, email)
        self.employee_id = employee_id
        self.department  = department

    def badge(self):
        return f"{self.employee_id} — {self.full_name}"

    def __str__(self):
        return f"[{self.employee_id}] {self.full_name} — {self.department}"

emp = Employee(
    "Hanu", "Madala", "20 Jan 2002", "+65 1234 5678", "hanu.m@tdfs.com",
    "EMP-001", "Engineering"
)
print(emp.badge())
print(str(emp))
```

</details>

In [ ]:
# Exercise 4 — your code here